In [ ]:
import gpytoolbox as gpy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.collections as mc
import matplotlib.patches as mp
import trimesh

import dvx_ext

# 3D Voxelization Functions Overview

## Helper Functions

In [ ]:
mesh = trimesh.load_mesh("hand.obj")
V, F = np.array(mesh.vertices, dtype=np.float32), np.array(mesh.faces, dtype=np.int32)

# Global parameters
GRID_SIZE = (64, 64, 64)
FILTER_RADIUS = (2 / GRID_SIZE[0]) / 2
NUM_SAMPLES_PER_VOXEL = 64
NUM_SAMPLES_PER_SIMPLEX = 64
FD_EPSILON = 1e-4

print(f"Mesh info: {len(V)} vertices, {len(F)} faces")
print(f"Grid size: {GRID_SIZE}")
print(f"Filter radius: {FILTER_RADIUS}")

In [ ]:
def test_primal(func, dtype, method_name, **kwargs):
    v = V.astype(dtype)
    f = F.astype(np.uint32)
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)
    
    # Run once to get result
    func(v, f, occupancy, **kwargs)
    
    print(f"{method_name}:")
    print(f"  Output range: [{occupancy.min():.6f}, {occupancy.max():.6f}]")

    return occupancy


def test_forward_ad(primal_func, forward_func, dtype, method_name, primal_kwargs, forward_kwargs):
    v = V.astype(dtype)
    f = F.astype(np.uint32)
    
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)

    # Random perturbation in vertex space
    d_vertices = np.random.randn(*v.shape).astype(dtype) * 0.01
    
    # Forward AD
    d_occupancy_ad = np.zeros(GRID_SIZE, dtype=dtype)
    forward_func(v, f, occupancy, d_vertices, d_occupancy_ad, **forward_kwargs)
    
    # Finite differences
    v_minus = v - FD_EPSILON * d_vertices
    occupancy_minus = np.zeros(GRID_SIZE, dtype=dtype)
    primal_func(v_minus, f, occupancy_minus, **primal_kwargs)

    v_plus = v + FD_EPSILON * d_vertices
    occupancy_plus = np.zeros(GRID_SIZE, dtype=dtype)
    primal_func(v_plus, f, occupancy_plus, **primal_kwargs)
    d_occupancy_fd = (occupancy_plus - occupancy_minus) / (2 * FD_EPSILON)
    
    # Compare
    error = np.abs(d_occupancy_ad - d_occupancy_fd).mean() / (np.abs(d_occupancy_fd).max() + 1e-10)

    print(f"{method_name} (Forward AD):")
    print(f"  Relative error vs FD: {error:.6e}")
    
    return error



def test_backward_ad(primal_func, backward_func, dtype, method_name, primal_kwargs, backward_kwargs):
    v = V.astype(dtype)
    f = F.astype(np.uint32)
    
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)
    d_vertices_ad = np.zeros_like(v)

    # Random perturbation in vertex space
    d_vertices = np.random.randn(*v.shape).astype(dtype)

    # Random gradient in occupancy space - this defines our gradient with respect to some Loss
    d_L = np.ones(GRID_SIZE).astype(dtype)
    d_L = np.zeros(GRID_SIZE).astype(dtype)
    d_L[0, 0, 0] = 1.0
    # d_L = np.random.randn(*GRID_SIZE).astype(dtype)
    
    # Finite differences
    v_minus = v - FD_EPSILON * d_vertices
    occupancy_minus = np.zeros(GRID_SIZE, dtype=dtype)
    primal_func(v_minus, f, occupancy_minus, **primal_kwargs)

    v_plus = v + FD_EPSILON * d_vertices
    occupancy_plus = np.zeros(GRID_SIZE, dtype=dtype)
    primal_func(v_plus, f, occupancy_plus, **primal_kwargs)
    d_occupancy_fd = (occupancy_plus - occupancy_minus) / (2 * FD_EPSILON)
    
    # gradient of L w.r.t. direction d_vertices via FD
    dL_dd_fd = (d_occupancy_fd * d_L).sum()

    # Backward AD
    backward_func(v, f, occupancy, d_vertices_ad, d_L, **backward_kwargs)

    # Directional derivative via AD
    dL_dd_ad = np.sum(d_vertices * d_vertices_ad)

    # Relative error
    error = abs(dL_dd_fd - dL_dd_ad) / (np.abs(dL_dd_fd).max() + 1e-10)
    
    print(f"{method_name} (Backward AD):")
    print(f"  Directional deriv (FD): {dL_dd_fd:.6e}")
    print(f"  Directional deriv (AD): {dL_dd_ad:.6e}")
    print(f"  Relative error: {error:.6e}")
    
    return error

## Closed Form Method

### float32

In [ ]:
# Primal
occupancy_cf_f32 = test_primal(dvx_ext.voxelize_cf_f32, np.float32, "voxelize_cf_f32")

import helper
import polyscope as ps

ps.init()
helper.render_grid("data", occupancy_cf_f32)
ps.show()

In [ ]:
# Forward AD
test_forward_ad(dvx_ext.voxelize_cf_f32, dvx_ext.voxelize_forward_cf_f32, np.float32, 
                "voxelize_forward_cf_f32", {}, {})

In [ ]:
# Backward AD
test_backward_ad(dvx_ext.voxelize_cf_f32, dvx_ext.voxelize_backward_cf_f32, np.float32,
                 "voxelize_backward_cf_f32", {}, {})

### float64

In [ ]:
# Primal
occupancy_cf_f64 = test_primal(dvx_ext.voxelize_cf_f64, np.float64, "voxelize_cf_f64")

In [ ]:
# Forward AD
test_forward_ad(dvx_ext.voxelize_cf_f64, dvx_ext.voxelize_forward_cf_f64, np.float64,
                "voxelize_forward_cf_f64", {}, {})

In [ ]:
# Backward AD
test_backward_ad(dvx_ext.voxelize_cf_f64, dvx_ext.voxelize_backward_cf_f64, np.float64,
                 "voxelize_backward_cf_f64", {}, {})

## Monte Carlo Method

### float32

In [ ]:
# Primal
mc_kwargs = {'num_samples_per_voxel': NUM_SAMPLES_PER_VOXEL, 'filter_radius': FILTER_RADIUS}
occupancy_mc_f32 = test_primal(dvx_ext.voxelize_mc_f32, np.float32, "voxelize_mc_f32", **mc_kwargs)

In [ ]:
# Forward AD
mc_forward_kwargs = {'num_samples_per_simplex': NUM_SAMPLES_PER_SIMPLEX, 'filter_radius': FILTER_RADIUS}
test_forward_ad(dvx_ext.voxelize_mc_f32, dvx_ext.voxelize_forward_mc_f32, np.float32,
                "voxelize_forward_mc_f32", mc_kwargs, mc_forward_kwargs)

In [ ]:
# Backward AD
mc_backward_kwargs = {'num_samples_per_simplex': NUM_SAMPLES_PER_SIMPLEX, 'filter_radius': FILTER_RADIUS}
test_backward_ad(dvx_ext.voxelize_mc_f32, dvx_ext.voxelize_backward_mc_f32, np.float32,
                 "voxelize_backward_mc_f32", mc_kwargs, mc_backward_kwargs)

### float64

In [ ]:
# Primal
occupancy_mc_f64 = test_primal(dvx_ext.voxelize_mc_f64, np.float64, "voxelize_mc_f64", **mc_kwargs)

In [ ]:
# Forward AD
test_forward_ad(dvx_ext.voxelize_mc_f64, dvx_ext.voxelize_forward_mc_f64, np.float64,
                "voxelize_forward_mc_f64", mc_kwargs, mc_forward_kwargs)

In [ ]:
# Backward AD
test_backward_ad(dvx_ext.voxelize_mc_f64, dvx_ext.voxelize_backward_mc_f64, np.float64,
                 "voxelize_backward_mc_f64", mc_kwargs, mc_backward_kwargs)